# Notebook 06: Rolling Forecast Evaluation

This notebook compares ARIMA, ARIMAX, and VAR using repeated expanding-window forecasts. A horizon of one month means predicting the next observation; a horizon of six months means predicting six observations ahead. Errors are summarized with MAE, RMSE, and MAPE, and Diebold-Mariano tests compare squared-error loss between model pairs.

# Forecast evaluation

Compare ARIMA, ARIMAX, and VAR using rolling-origin forecasts at horizons 1-6 months. MAE, RMSE, MAPE, and Diebold-Mariano tests are calculated from repeated out-of-sample forecast errors.

In [1]:
import warnings

import numpy as np
import pandas as pd
from scipy.stats import norm
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.api import VAR

warnings.filterwarnings("ignore")


In [2]:
df = pd.read_csv("../data/dataset_final/full_data.csv")
df["date"] = pd.to_datetime(df["month"], format="%YM%m")
df = df.set_index("date").asfreq("MS")

fuel_cols = [
    "diesel_price_(GHC/litre)",
    "petrol_price_ghs/litre",
    "lpg_ghs/kg",
]
external_cols = [
    "brent_price_usd_per_barrel",
    "usd/ghs_rate",
]
var_cols = fuel_cols + external_cols + ["consumer_price_index"]
max_horizon = 6
initial_train_size = 96
candidate_orders = [(p, 1, q) for p in range(4) for q in range(4)]

# Select fixed model orders using only the first training window.
initial_train = df.iloc[:initial_train_size]

In [3]:
def select_arima_order(series, exog=None):
    best_order = None
    best_aic = float("inf")
    for order in candidate_orders:
        try:
            fit = ARIMA(
                series,
                exog=exog,
                order=order,
                enforce_stationarity=False,
            ).fit()
            if fit.aic < best_aic:
                best_order, best_aic = order, fit.aic
        except (ValueError, np.linalg.LinAlgError):
            continue
    return best_order

arima_orders = {
    column: select_arima_order(initial_train[column]) for column in fuel_cols
}
arimax_orders = {
    column: select_arima_order(
        initial_train[column], initial_train[external_cols]
    )
    for column in fuel_cols
}

# Select the best stable second-difference VAR lag from the initial window.
initial_var_diff = initial_train[var_cols].diff().diff().dropna()
var_model = VAR(initial_var_diff)
lag_selection = var_model.select_order(maxlags=6)
stable_var_candidates = []
for lag in range(1, 7):
    try:
        candidate_fit = var_model.fit(lag)
        if candidate_fit.is_stable():
            stable_var_candidates.append((candidate_fit.aic, lag))
    except (ValueError, np.linalg.LinAlgError):
        continue

if stable_var_candidates:
    selected_var_lag = min(stable_var_candidates)[1]
else:
    selected_var_lag = max(1, min(lag_selection.selected_orders.get("bic") or 1, 6))

print("ARIMA orders:", arima_orders)
print("ARIMAX orders:", arimax_orders)
print("VAR lag:", selected_var_lag)

ARIMA orders: {'diesel_price_(GHC/litre)': (2, 1, 2), 'petrol_price_ghs/litre': (0, 1, 3), 'lpg_ghs/kg': (3, 1, 3)}
ARIMAX orders: {'diesel_price_(GHC/litre)': (1, 1, 2), 'petrol_price_ghs/litre': (0, 1, 3), 'lpg_ghs/kg': (3, 1, 1)}
VAR lag: 5


In [4]:
# Store repeated forecast errors for every model, series, and horizon.
errors = {
    model_name: {
        column: {horizon: [] for horizon in range(1, max_horizon + 1)}
        for column in fuel_cols
    }
    for model_name in ["ARIMA", "ARIMAX", "VAR"]
}

origins = range(initial_train_size, len(df) - max_horizon + 1)
for origin in origins:
    history = df.iloc[:origin]
    future = df.iloc[origin : origin + max_horizon]

    for column in fuel_cols:
        arima_fit = ARIMA(
            history[column],
            order=arima_orders[column],
            enforce_stationarity=False,
        ).fit()
        arima_forecast = arima_fit.forecast(steps=max_horizon)

        arimax_fit = ARIMA(
            history[column],
            exog=history[external_cols],
            order=arimax_orders[column],
            enforce_stationarity=False,
        ).fit()
        arimax_forecast = arimax_fit.forecast(
            steps=max_horizon,
            exog=future[external_cols],
        )

        for horizon in range(1, max_horizon + 1):
            actual = future[column].iloc[horizon - 1]
            errors["ARIMA"][column][horizon].append(
                actual - arima_forecast.iloc[horizon - 1]
            )
            errors["ARIMAX"][column][horizon].append(
                actual - arimax_forecast.iloc[horizon - 1]
            )

    # VAR uses second differences because CPI was not stationary after d=1.
    history_diff = history[var_cols].diff().diff().dropna()
    var_fit = VAR(history_diff).fit(selected_var_lag)
    var_diff_forecast = var_fit.forecast(
        history_diff.values[-selected_var_lag:],
        steps=max_horizon,
    )
    last_level = history[var_cols].iloc[-1]
    last_first_difference = history[var_cols].diff().dropna().iloc[-1]
    var_first_difference_forecast = (
        last_first_difference.to_numpy() + np.cumsum(var_diff_forecast, axis=0)
    )
    var_level_forecast = last_level.to_numpy() + np.cumsum(
        var_first_difference_forecast, axis=0
    )
    var_level_forecast = pd.DataFrame(
        var_level_forecast,
        index=future.index,
        columns=var_cols,
    )

    for column in fuel_cols:
        for horizon in range(1, max_horizon + 1):
            actual = future[column].iloc[horizon - 1]
            errors["VAR"][column][horizon].append(
                actual - var_level_forecast[column].iloc[horizon - 1]
            )

# Convert errors into accuracy measures by horizon.
metric_rows = []
for model_name in errors:
    for column in fuel_cols:
        for horizon in range(1, max_horizon + 1):
            error = np.asarray(errors[model_name][column][horizon], dtype=float)
            actual = df[column].iloc[
                initial_train_size + horizon - 1 : len(df) - max_horizon + horizon
            ].to_numpy()
            metric_rows.append(
                {
                    "model": model_name,
                    "series": column,
                    "horizon_months": horizon,
                    "MAE": np.mean(np.abs(error)),
                    "RMSE": np.sqrt(np.mean(error**2)),
                    "MAPE": np.mean(np.abs(error / actual)) * 100,
                    "origins": len(error),
                }
            )

metrics = pd.DataFrame(metric_rows)
print("Rolling-origin forecast accuracy:")
print(metrics.to_string(index=False))

Rolling-origin forecast accuracy:
 model                   series  horizon_months      MAE     RMSE      MAPE  origins
 ARIMA diesel_price_(GHC/litre)               1 0.386753 0.546911  3.007630       31
 ARIMA diesel_price_(GHC/litre)               2 0.564236 0.712213  4.297732       31
 ARIMA diesel_price_(GHC/litre)               3 0.714129 0.895667  5.416125       31
 ARIMA diesel_price_(GHC/litre)               4 0.856438 1.066994  6.453925       31
 ARIMA diesel_price_(GHC/litre)               5 0.915860 1.137451  6.894572       31
 ARIMA diesel_price_(GHC/litre)               6 1.015895 1.228644  7.635808       31
 ARIMA   petrol_price_ghs/litre               1 0.334694 0.414689  2.447408       31
 ARIMA   petrol_price_ghs/litre               2 0.532353 0.657943  3.862232       31
 ARIMA   petrol_price_ghs/litre               3 0.672972 0.834720  4.830052       31
 ARIMA   petrol_price_ghs/litre               4 0.857922 1.019370  6.108616       31
 ARIMA   petrol_price_ghs/litre

In [5]:
# Diebold-Mariano test using a HAC variance for the loss differential.
def diebold_mariano(errors_one, errors_two, horizon):
    loss_difference = np.asarray(errors_one) ** 2 - np.asarray(errors_two) ** 2
    sample_size = len(loss_difference)
    mean_difference = loss_difference.mean()
    centered = loss_difference - mean_difference
    bandwidth = min(max(horizon - 1, 0), sample_size - 1)
    variance = np.mean(centered**2)
    for lag in range(1, bandwidth + 1):
        weight = 1 - lag / (bandwidth + 1)
        covariance = np.mean(centered[lag:] * centered[:-lag])
        variance += 2 * weight * covariance
    variance = max(variance, np.finfo(float).eps)
    statistic = mean_difference / np.sqrt(variance / sample_size)
    p_value = 2 * norm.sf(abs(statistic))
    return statistic, p_value

dm_rows = []
comparisons = [("ARIMA", "ARIMAX"), ("ARIMA", "VAR"), ("ARIMAX", "VAR")]
for column in fuel_cols:
    for horizon in range(1, max_horizon + 1):
        for model_one, model_two in comparisons:
            statistic, p_value = diebold_mariano(
                errors[model_one][column][horizon],
                errors[model_two][column][horizon],
                horizon,
            )
            dm_rows.append(
                {
                    "series": column,
                    "horizon_months": horizon,
                    "model_one": model_one,
                    "model_two": model_two,
                    "DM_statistic": statistic,
                    "p_value": p_value,
                }
            )

dm_results = pd.DataFrame(dm_rows)
print("Diebold-Mariano tests using squared-error loss:")
print(dm_results.to_string(index=False))

Diebold-Mariano tests using squared-error loss:
                  series  horizon_months model_one model_two  DM_statistic  p_value
diesel_price_(GHC/litre)               1     ARIMA    ARIMAX     -0.928012 0.353401
diesel_price_(GHC/litre)               1     ARIMA       VAR     -2.254562 0.024161
diesel_price_(GHC/litre)               1    ARIMAX       VAR     -1.921409 0.054680
diesel_price_(GHC/litre)               2     ARIMA    ARIMAX     -1.284227 0.199063
diesel_price_(GHC/litre)               2     ARIMA       VAR     -2.810904 0.004940
diesel_price_(GHC/litre)               2    ARIMAX       VAR     -2.326354 0.020000
diesel_price_(GHC/litre)               3     ARIMA    ARIMAX     -0.849226 0.395756
diesel_price_(GHC/litre)               3     ARIMA       VAR     -3.407201 0.000656
diesel_price_(GHC/litre)               3    ARIMAX       VAR     -2.894304 0.003800
diesel_price_(GHC/litre)               4     ARIMA    ARIMAX     -0.848191 0.396332
diesel_price_(GHC/litre)    

In [6]:
# Cumulative evaluation required by the project methodology.
# Accuracy metrics pool horizons 1 through 1, 3, or 6 months.
cumulative_horizons = [1, 3, 6]
cumulative_metric_rows = []
cumulative_dm_rows = []

for cutoff in cumulative_horizons:
    for model_name in errors:
        for column in fuel_cols:
            pooled_errors = np.concatenate(
                [errors[model_name][column][step] for step in range(1, cutoff + 1)]
            )
            pooled_actuals = np.concatenate(
                [
                    df[column].iloc[
                        initial_train_size + step - 1 : len(df) - max_horizon + step
                    ].to_numpy()
                    for step in range(1, cutoff + 1)
                ]
            )
            cumulative_metric_rows.append(
                {
                    "model": model_name,
                    "series": column,
                    "cumulative_horizon_months": cutoff,
                    "MAE": np.mean(np.abs(pooled_errors)),
                    "RMSE": np.sqrt(np.mean(pooled_errors**2)),
                    "MAPE": np.mean(np.abs(pooled_errors / pooled_actuals)) * 100,
                    "forecast_cases": len(pooled_errors),
                }
            )

In [ ]:
# For cumulative DM tests, preserve the rolling-origin order by comparing
# each origin's average loss over the first 1, 3, or 6 forecast steps.
def diebold_mariano_losses(loss_one, loss_two, bandwidth):
    loss_difference = np.asarray(loss_one) - np.asarray(loss_two)
    sample_size = len(loss_difference)
    mean_difference = loss_difference.mean()
    centered = loss_difference - mean_difference
    bandwidth = min(bandwidth, sample_size - 1)
    variance = np.mean(centered**2)
    for lag in range(1, bandwidth + 1):
        weight = 1 - lag / (bandwidth + 1)
        covariance = np.mean(centered[lag:] * centered[:-lag])
        variance += 2 * weight * covariance
    variance = max(variance, np.finfo(float).eps)
    statistic = mean_difference / np.sqrt(variance / sample_size)
    p_value = 2 * norm.sf(abs(statistic))
    return statistic, p_value

for cutoff in cumulative_horizons:
    for column in fuel_cols:
        for model_one, model_two in comparisons:
            loss_one = np.mean(
                [
                    np.asarray(errors[model_one][column][step]) ** 2
                    for step in range(1, cutoff + 1)
                ],
                axis=0,
            )
            loss_two = np.mean(
                [
                    np.asarray(errors[model_two][column][step]) ** 2
                    for step in range(1, cutoff + 1)
                ],
                axis=0,
            )
            statistic, p_value = diebold_mariano_losses(
                loss_one,
                loss_two,
                cutoff - 1,
            )
            cumulative_dm_rows.append(
                {
                    "series": column,
                    "cumulative_horizon_months": cutoff,
                    "model_one": model_one,
                    "model_two": model_two,
                    "DM_statistic": statistic,
                    "p_value": p_value,
                }
            )

cumulative_metrics = pd.DataFrame(cumulative_metric_rows)
cumulative_dm_results = pd.DataFrame(cumulative_dm_rows)
print("Cumulative forecast accuracy for horizons 1, 3, and 6 months:")
print(cumulative_metrics.to_string(index=False))
print("\nCumulative Diebold-Mariano tests:")
print(cumulative_dm_results.to_string(index=False))

Cumulative forecast accuracy for horizons 1, 3, and 6 months:
 model                   series  cumulative_horizon_months      MAE     RMSE      MAPE  forecast_cases
 ARIMA diesel_price_(GHC/litre)                          1 0.386753 0.546911  3.007630              31
 ARIMA   petrol_price_ghs/litre                          1 0.334694 0.414689  2.447408              31
 ARIMA               lpg_ghs/kg                          1 0.185263 0.252495  1.305504              31
ARIMAX diesel_price_(GHC/litre)                          1 0.523495 0.620796  4.012037              31
ARIMAX   petrol_price_ghs/litre                          1 0.517954 0.636771  3.721674              31
ARIMAX               lpg_ghs/kg                          1 0.207277 0.268194  1.450945              31
   VAR diesel_price_(GHC/litre)                          1 0.929998 1.275101  7.114103              31
   VAR   petrol_price_ghs/litre                          1 1.312347 1.764460  9.550780              31
   VAR     